# OT/ICS-IDS — Anchor Max-Context Sensitivity (v1.4)

**Companion paper:** *A Comparative Study of Large Language Models for Industrial Cyber-Physical Security* (de Curtò, de Zarzà, Cano, Calafate, *Electronics* 2026)

## Purpose

The paper's E7 headline protocol trains every method (LLM, TabPFN, TabICL, RandomForest, XGBoost) on a **K-shot ICL split** of 10 examples per class. This is the only regime that all six families can support uniformly, because open-source LLMs have hard context-window limits.

But tabular foundation models (TabPFN, TabICL) and classical methods (RF, XGBoost) can actually ingest substantially more training data than the K-shot protocol allows:

- **TabPFN v2** supports up to ~10,000 support rows in a single in-context call.
- **TabICL** supports ~60,000+ support rows in standard config.
- **RF / XGBoost** scale to arbitrary sample sizes.

This notebook runs a **max-context sensitivity analysis**: each anchor is evaluated at *both* the paper's K-shot budget and its native maximum budget. The comparison exposes whether the K-shot ranking reported in the paper transfers to a regime where tabular methods are not data-constrained, and quantifies the cost of the K-shot constraint on each method family.

LLMs are not re-run because K=10 is their structural ceiling.

## Two sweeps

**Sweep A — K-shot regime.** Same as the paper's E7 protocol: 10 examples per class for every anchor, identical to v8 and v1.3.

**Sweep B — Max-context regime.** Each anchor at its native budget:

| Method | Train budget (Sweep B) |
|---|---|
| RandomForest | Full 80% training partition (~80k SWaT, ~50k HAI, ~80k WUSTL) |
| XGBoost | Full 80% training partition (same as RF) |
| TabPFN | 10,000 stratified support rows (v2 native limit) |
| TabICL | 50,000 stratified support rows (native large-context support) |

Both sweeps evaluate on the **same 20% stratified holdout**, so the only difference between Sweep A and Sweep B rows is the training-data budget.

## Protocol parity

- **Datasets:** SWaT, HAI, WUSTL-IIoT-2021 (binary + 5-class on WUSTL)
- **Pre-processing:** v8-verbatim loaders, label resolvers, 50k-per-class subsampler
- **Feature selection:** Top-12 features by mutual information against the label (v8 cell 12)
- **Split:** Stratified 80%/20% on the processed corpus → `train_pool` and `holdout`
- **Seeds:** 42, 43, 44 (3 seeds, matching the paper's E7 protocol)
- **Metrics:** Accuracy, Macro F1, MCC, FAR, DR, per-class F1
- **TabPFN access:** `tabpfn-client` cloud API (matches the user's setup)
- **TabICL access:** Local `tabicl` package (Colab A100, matches the user's setup)

## Outputs

CSVs written to `./xgb_outputs_v1_4/`:

- `v14_sweep_A_kshot.csv` — per-(model, seed, dataset, task) rows for the K-shot regime
- `v14_sweep_B_maxctx.csv` — same but max-context regime
- `v14_combined.csv` — both sweeps stacked with a `regime` column
- `v14_delta.csv` — mean delta (max-context minus K-shot) per (model, dataset, task)
- `v14_perclass_wustl.csv` — per-class F1 for WUSTL multi-class across regimes and seeds

These feed the §5.7 "Max-context sensitivity" paragraph in the manuscript.

## Cost (estimated, A100)

- RF on full training partition: ~30s per fit × 12 = ~6 min
- XGBoost on full training partition: ~30s per fit × 12 = ~6 min
- TabPFN at 10k support (cloud API): ~30-60s per call × 12 = ~10 min
- TabICL at 50k support (A100 GPU): ~30-60s per call × 12 = ~10 min
- Plus K-shot regime for all 4 methods: ~5 min total
- Plus Kaggle data download: ~15 min
- **Total end-to-end: 45-75 minutes**

No LLM API spend (LLMs not re-run).


In [1]:
# ============================================================
# 0) INSTALL & IMPORTS
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        __import__(name)
    except ImportError:
        print(f"  installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("xgboost")
_ensure("kagglehub")
_ensure("tabpfn-client", "tabpfn_client")
_ensure("tabicl")

import os, gc, glob, json, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, matthews_corrcoef,
    precision_recall_fscore_support,
)

# Optional imports (soft-fail so RF/XGBoost-only runs still work)
HAS_TABPFN = False
HAS_TABICL = False
try:
    from tabpfn_client import init as tabpfn_init, TabPFNClassifier as TabPFNCloud
    HAS_TABPFN = True
    print("  tabpfn-client: OK")
except ImportError:
    print("  tabpfn-client: NOT AVAILABLE (will skip TabPFN runs)")

try:
    import torch
    from tabicl import TabICLClassifier
    HAS_TABICL = True
    print(f"  tabicl:        OK  (torch CUDA: {torch.cuda.is_available()})")
    if torch.cuda.is_available():
        print(f"  GPU:           {torch.cuda.get_device_name(0)}")
except ImportError as e:
    print(f"  tabicl:        NOT AVAILABLE (will skip TabICL runs): {e}")

warnings.filterwarnings("ignore")
print()
print(f"xgboost version: {xgb.__version__}")
print(f"numpy   version: {np.__version__}")
print(f"pandas  version: {pd.__version__}")


  installing tabpfn-client...
  installing tabicl...
  tabpfn-client: OK
  tabicl:        OK  (torch CUDA: True)
  GPU:           NVIDIA A100-SXM4-40GB

xgboost version: 3.2.0
numpy   version: 2.0.2
pandas  version: 2.2.2


In [2]:
# ============================================================
# 1) KAGGLE AUTH (Colab Secrets first, env-var second)
# ============================================================
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
    print("  Kaggle auth: Colab Secrets")
except Exception:
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        print("  Kaggle auth: env vars OK")
    else:
        print("  WARNING: no Kaggle auth found. Set KAGGLE_USERNAME and KAGGLE_KEY")


  Kaggle auth: Colab Secrets


In [3]:
# ============================================================
# 2) DATASET CONFIG + BUDGETS
# ============================================================
DATASET_SLUGS = {
    "swat" : "vishala28/swat-dataset-secure-water-treatment-system",
    "hai"  : "icsdataset/hai-security-dataset",
    "wustl": "annaamalaiu/wustl-iiot-2021-dataset",
}

HAI_VERSION_PREFERENCE = ["hai-22.04", "hai-21.03", "hai-20.07",
                          "hai-1.0", "hai-2.0", "hai-3.0"]

SEED_DEFAULT         = 42
E7_N_SEEDS           = 3
SEED_LIST            = list(range(SEED_DEFAULT, SEED_DEFAULT + E7_N_SEEDS))

SUBSAMPLE_PER_CLASS  = 50_000   # corpus cap (v8)
K_SHOT_PER_CLASS     = 10       # Sweep A training budget per class (matches v8 E7_K_SHOT)
TRAIN_FRAC           = 0.80     # stratified train/holdout split
TOP_K_FEATURES       = 12       # MI feature selection (v8 cell 12)

# Max-context budgets for Sweep B
MAX_CTX_TABPFN       = 10_000   # TabPFN v2 native single-context limit
MAX_CTX_TABICL       = 50_000   # TabICL native large-context support
# RF and XGBoost have no upper bound -- they use the full train_pool

NORMAL_LABEL         = "Normal"

OUT_DIR = Path("./xgb_outputs_v1_4"); OUT_DIR.mkdir(exist_ok=True)
print(f"Output directory:     {OUT_DIR.resolve()}")
print(f"Seeds:                {SEED_LIST}")
print(f"K_SHOT_PER_CLASS:     {K_SHOT_PER_CLASS}")
print(f"MAX_CTX_TABPFN:       {MAX_CTX_TABPFN:,}")
print(f"MAX_CTX_TABICL:       {MAX_CTX_TABICL:,}")
print(f"TRAIN_FRAC:           {TRAIN_FRAC}")
print(f"SUBSAMPLE_PER_CLASS:  {SUBSAMPLE_PER_CLASS:,}  (corpus cap)")


Output directory:     /content/xgb_outputs_v1_4
Seeds:                [42, 43, 44]
K_SHOT_PER_CLASS:     10
MAX_CTX_TABPFN:       10,000
MAX_CTX_TABICL:       50,000
TRAIN_FRAC:           0.8
SUBSAMPLE_PER_CLASS:  50,000  (corpus cap)


In [4]:
# ============================================================
# 3) DATA LOADER (v8 verbatim)
# ============================================================
def _download_kaggle(slug: str) -> str:
    try:
        import kagglehub
        folder = kagglehub.dataset_download(slug)
        print(f"  kagglehub OK -> {folder}")
        return folder
    except Exception as e:
        print(f"  kagglehub failed ({type(e).__name__}: {e}); falling back to kaggle CLI...")
        target = os.path.join("./.kaggle_cache", slug.replace("/", "__"))
        os.makedirs(target, exist_ok=True)
        kg_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kg_dir, exist_ok=True)
        kg_json = os.path.join(kg_dir, "kaggle.json")
        if not os.path.exists(kg_json):
            with open(kg_json, "w") as f:
                json.dump({"username": os.environ["KAGGLE_USERNAME"],
                           "key":      os.environ["KAGGLE_KEY"]}, f)
            os.chmod(kg_json, 0o600)
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
        except ImportError:
            os.system("pip install -q kaggle")
            from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi(); api.authenticate()
        api.dataset_download_files(slug, path=target, unzip=True, quiet=False)
        return target


def _read_swat(folder: str) -> pd.DataFrame:
    csvs = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
    csvs = [c for c in csvs if "_macosx" not in c.lower()]
    print(f"  found {len(csvs)} CSV file(s): {[os.path.basename(c) for c in csvs]}")
    if not csvs: raise FileNotFoundError(folder)
    def _try_read(path):
        for sep in (",", ";"):
            for enc in ("utf-8", "latin-1"):
                try:
                    df = pd.read_csv(path, sep=sep, encoding=enc, low_memory=False)
                    if df.shape[1] >= 5: return df
                except Exception: continue
        return pd.read_csv(path, sep=";", decimal=",", encoding="latin-1", low_memory=False)
    dfs = []
    for c in csvs:
        df = _try_read(c)
        df["_source_file"] = os.path.basename(c)
        dfs.append(df)
        print(f"    {os.path.basename(c):<50} -> {df.shape}")
    return pd.concat(dfs, ignore_index=True)


def _read_hai(folder: str) -> pd.DataFrame:
    subdirs = [d for d in glob.glob(os.path.join(folder, "*")) if os.path.isdir(d)]
    chosen = None
    for pref in HAI_VERSION_PREFERENCE:
        for d in subdirs:
            if pref in os.path.basename(d).lower().replace("_", "-"):
                chosen = d; break
        if chosen: break
    if chosen is None: chosen = folder
    print(f"  HAI version chosen: {os.path.basename(chosen)}")
    csvs = sorted(glob.glob(os.path.join(chosen, "**", "*.csv"), recursive=True))
    csvs = [c for c in csvs if "_macosx" not in c.lower()]
    csvs = [c for c in csvs if "haiend" not in os.path.basename(c).lower()
                              and "hai-end" not in os.path.basename(c).lower()]
    if not csvs: raise FileNotFoundError(chosen)
    print(f"  reading {len(csvs)} HAI CSV file(s)")
    dfs = []
    for c in csvs:
        df = pd.read_csv(c, low_memory=False)
        df["_source_file"] = os.path.basename(c)
        dfs.append(df)
        print(f"    {os.path.basename(c):<30} -> {df.shape}")
    return pd.concat(dfs, ignore_index=True)


def _read_wustl(folder: str) -> pd.DataFrame:
    csvs = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
    csvs = [c for c in csvs if "_macosx" not in c.lower()]
    if not csvs: raise FileNotFoundError(folder)
    print(f"  found {len(csvs)} CSV file(s): {[os.path.basename(c) for c in csvs]}")
    dfs = []
    for c in csvs:
        df = pd.read_csv(c, low_memory=False)
        df["_source_file"] = os.path.basename(c)
        dfs.append(df)
        print(f"    {os.path.basename(c):<40} -> {df.shape}")
    return pd.concat(dfs, ignore_index=True)


def load_dataset_raw(name: str) -> pd.DataFrame:
    slug = DATASET_SLUGS[name]
    print(f"\n[load] {name} ({slug})")
    folder = _download_kaggle(slug)
    if name == "swat":   raw = _read_swat(folder)
    elif name == "hai":  raw = _read_hai(folder)
    elif name == "wustl": raw = _read_wustl(folder)
    else: raise ValueError(name)
    raw.columns = [c.strip() for c in raw.columns]
    print(f"  raw {name}: {raw.shape}")
    return raw


In [5]:
# ============================================================
# 4) PREPROCESSORS (v8 verbatim) + 50k-per-class subsample
# ============================================================
SWAT_LABEL_CANDIDATES = ["Normal/Attack", "Normal_Attack", "label", "Label", "Attack"]
HAI_LABEL_CANDIDATES  = ["Attack", "attack", "label", "Label"]


def _find_label_column(df, candidates):
    for c in candidates:
        for col in df.columns:
            if col.strip().lower() == c.strip().lower():
                return col
    return None


def _basic_clean_numeric(df, label_col):
    df = df.copy()
    feature_cols = [c for c in df.columns if c != label_col and not c.startswith("_")]
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=feature_cols, how="any")
    for col in feature_cols:
        if pd.api.types.is_float_dtype(df[col]):
            cmin, cmax = df[col].min(), df[col].max()
            if cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
        elif pd.api.types.is_integer_dtype(df[col]):
            cmin, cmax = df[col].min(), df[col].max()
            if cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
    n_unique = df[feature_cols].nunique()
    constants = n_unique[n_unique <= 1].index.tolist()
    if constants:
        df = df.drop(columns=constants)
    return df


def preprocess_swat(df, binary=True):
    label_col = _find_label_column(df, SWAT_LABEL_CANDIDATES)
    if label_col is None:
        raise KeyError(f"No SWaT label in {list(df.columns)}")
    df = df.rename(columns={label_col: "Label"})
    drop_cols = [c for c in df.columns if c != "Label" and (
        c.startswith("_") or c.lower() in ("timestamp","time","datetime","date","unnamed: 0"))]
    if drop_cols: df = df.drop(columns=drop_cols)
    df["Label"] = df["Label"].astype(str).str.strip()
    df["Label"] = df["Label"].str.replace(r"\s+", "", regex=True)
    df["Label"] = df["Label"].str.replace(r"[^A-Za-z]", "", regex=True)
    df["Label"] = df["Label"].str.title()
    if binary:
        df.loc[df["Label"] != "Normal", "Label"] = "Attack"
    return _basic_clean_numeric(df, "Label")


def preprocess_hai(df, binary=True):
    label_col = _find_label_column(df, HAI_LABEL_CANDIDATES)
    if label_col is not None and binary:
        df = df.rename(columns={label_col: "_label_raw"})
        df["Label"] = df["_label_raw"].astype(str).str.strip().map(
            lambda v: "Normal" if v in ("0","0.0","Normal","normal","False","false","nan","") else "Attack"
        )
        df = df.drop(columns=["_label_raw"])
    else:
        per_proc_cols = [c for c in df.columns if c.lower().startswith("attack_p")]
        if not per_proc_cols:
            per_proc_cols = [c for c in df.columns if "attack" in c.lower()]
        if not per_proc_cols:
            raise KeyError(f"No HAI label in {list(df.columns)[:15]}")
        for c in per_proc_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df["Label"] = "Normal"
        attack_mask = (df[per_proc_cols].sum(axis=1) > 0)
        df.loc[attack_mask, "Label"] = "Attack"
        df = df.drop(columns=per_proc_cols)
    drop_cols = [c for c in df.columns if c != "Label" and (
        c.startswith("_") or c.lower() in ("time","timestamp","datetime","date","unnamed: 0"))]
    if drop_cols: df = df.drop(columns=drop_cols)
    return _basic_clean_numeric(df, "Label")


def _wustl_pick_binary_col(df, candidates):
    for col in candidates:
        if col not in df.columns: continue
        sample = df[col].dropna().head(5000)
        if len(sample) == 0: continue
        coerced = pd.to_numeric(sample, errors="coerce")
        ok = coerced.dropna()
        if len(ok) >= 0.99 * len(sample):
            uniq = set(ok.astype(int).unique().tolist())
            if uniq <= {0, 1} and len(uniq) >= 1:
                return col, {0: "Normal", 1: "Attack"}
        s = sample.astype(str).str.strip().str.lower()
        if set(s.unique()) <= {"normal", "attack"}:
            return col, {"normal": "Normal", "attack": "Attack"}
    return None


def _wustl_pick_categorical_col(df, candidates):
    for col in candidates:
        if col not in df.columns: continue
        sample = df[col].dropna().head(5000)
        if len(sample) == 0: continue
        coerced = pd.to_numeric(sample, errors="coerce")
        ok = coerced.dropna()
        if len(ok) < 0.5 * len(sample):
            return col
        uniq = set(ok.astype(int).unique().tolist()) if len(ok) > 0 else set()
        if len(uniq) > 2:
            return col
    return None


def preprocess_wustl(df, binary=True):
    LEAKAGE_COLS = ["StartTime","LastTime","SrcAddr","DstAddr","sIpId","dIpId"]
    leak_present = [c for c in df.columns if c.strip() in LEAKAGE_COLS]
    if leak_present:
        df = df.drop(columns=leak_present)
    BIN_CANDIDATES = ["Target","target","Traffic","traffic","Label","label"]
    CAT_CANDIDATES = ["Traffic","traffic","Label","label","Target","target"]
    bin_pick = _wustl_pick_binary_col(df, BIN_CANDIDATES)
    cat_pick = _wustl_pick_categorical_col(df, CAT_CANDIDATES)
    if binary:
        if bin_pick is not None:
            bin_col, mapping = bin_pick
            if 0 in mapping:
                df["Label"] = (pd.to_numeric(df[bin_col], errors="coerce")
                                  .fillna(0).astype(int).map(mapping))
            else:
                df["Label"] = (df[bin_col].astype(str).str.strip().str.lower()
                                  .map(mapping).fillna("Attack"))
        elif cat_pick is not None:
            df["Label"] = df[cat_pick].astype(str).str.strip().str.lower().map(
                lambda v: "Normal" if v in ("normal","0","0.0","false","","nan") else "Attack"
            )
        else:
            raise KeyError(f"No WUSTL binary label in {list(df.columns)[:20]}")
        for c in set(BIN_CANDIDATES + CAT_CANDIDATES):
            if c in df.columns and c != "Label":
                df = df.drop(columns=[c])
    else:
        if cat_pick is None:
            raise KeyError(f"No WUSTL categorical label in {list(df.columns)[:20]}")
        df = df.rename(columns={cat_pick: "_tmp_label"})
        df["Label"] = df["_tmp_label"].astype(str).str.strip().str.title()
        df.loc[df["Label"].str.lower() == "normal", "Label"] = "Normal"
        df = df.drop(columns=["_tmp_label"])
        for c in set(BIN_CANDIDATES + CAT_CANDIDATES):
            if c in df.columns and c != "Label":
                df = df.drop(columns=[c])
    drop_cols = [c for c in df.columns if c != "Label" and (
        c.startswith("_") or c.lower() in ("timestamp","time","datetime","date","unnamed: 0"))]
    if drop_cols: df = df.drop(columns=drop_cols)
    return _basic_clean_numeric(df, "Label")


def preprocess(name, df_raw, binary):
    if name == "swat":  return preprocess_swat(df_raw, binary=binary)
    if name == "hai":   return preprocess_hai(df_raw, binary=binary)
    if name == "wustl": return preprocess_wustl(df_raw, binary=binary)
    raise ValueError(name)


def subsample_per_class(data, cap=SUBSAMPLE_PER_CLASS, seed=0):
    """v8 random_state=0 per-class sample (matches the paper protocol exactly)."""
    if data["Label"].value_counts().max() <= cap:
        return data.reset_index(drop=True)
    return (data.groupby("Label", group_keys=False)
                .apply(lambda g: g.sample(min(len(g), cap), random_state=seed))
                .reset_index(drop=True))


In [6]:
# ============================================================
# 5) METRICS + FEATURE SELECTION (v8 verbatim)
# ============================================================
def metrics_from_predictions(y_true, y_pred, classes=None, normal_label=NORMAL_LABEL):
    y_true = np.asarray(y_true)
    y_pred = np.asarray([p if p is not None else "<NONE>" for p in y_pred])
    if classes is None:
        classes = sorted(set(y_true.tolist()))
    p, rec, f, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=classes, average=None, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    bal = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f.mean() if len(f) else 0.0
    try: mcc = matthews_corrcoef(y_true, y_pred)
    except: mcc = float("nan")
    normal_mask = (y_true == normal_label)
    far = float((y_pred[normal_mask] != normal_label).sum() / normal_mask.sum()) if normal_mask.sum() > 0 else float("nan")
    attack_mask = (y_true != normal_label)
    dr  = float((y_pred[attack_mask] != normal_label).sum() / attack_mask.sum()) if attack_mask.sum() > 0 else float("nan")
    per_class = {c: {"precision": float(p[i]), "recall": float(rec[i]),
                     "f1": float(f[i]), "support": int(sup[i])} for i, c in enumerate(classes)}
    return {"accuracy": float(acc), "balanced_accuracy": float(bal),
            "macro_f1": float(macro_f1), "mcc": float(mcc),
            "false_alarm_rate": far, "detection_rate": dr,
            "per_class": per_class, "classes": classes}


def select_top_features(df, k=TOP_K_FEATURES, sample=50_000, seed=0):
    """Top-k features by MI against the Label. v8 cell 12 verbatim."""
    rng = np.random.default_rng(seed)
    if len(df) > sample:
        idx = rng.choice(len(df), size=sample, replace=False)
        sub = df.iloc[idx]
    else:
        sub = df
    drop_cols = [c for c in sub.columns if c == "Label" or c.startswith("_")]
    X = sub.drop(columns=drop_cols).select_dtypes(include=[np.number])
    y = LabelEncoder().fit_transform(sub["Label"])
    mi = mutual_info_classif(X, y, random_state=seed)
    order = np.argsort(mi)[::-1]
    return list(X.columns[order[:k]])


In [7]:
# ============================================================
# 6) SPLIT HELPERS
# ============================================================
def build_kshot_split(df, features, k_per_class, holdout_df, seed=0):
    """K-shot training set: K examples per class drawn from `df` (not from
    holdout_df). Returns icl_df with K_per_class * n_classes rows.

    Selects K-per-class deterministically by random_state=seed permutation.
    """
    cols = features + ["Label"]
    rng = np.random.default_rng(seed)
    rows = []
    for cls, sub in df.groupby("Label"):
        idx = rng.permutation(len(sub))
        n_take = min(k_per_class, len(sub))
        rows.append(sub.iloc[idx[:n_take]][cols])
    return (pd.concat(rows, ignore_index=True)
              .sample(frac=1, random_state=seed)
              .reset_index(drop=True))


def stratified_split(data, train_frac=TRAIN_FRAC, seed=0):
    """Stratified 80/20 split: returns (train_pool, holdout) with class
    balance preserved in each partition."""
    train_rows, hold_rows = [], []
    rng = np.random.default_rng(seed)
    for cls, sub in data.groupby("Label"):
        n = len(sub)
        n_train = int(round(train_frac * n))
        idx = rng.permutation(n)
        sub2 = sub.iloc[idx]
        train_rows.append(sub2.iloc[:n_train])
        hold_rows.append(sub2.iloc[n_train:])
    train = pd.concat(train_rows, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    hold  = pd.concat(hold_rows,  ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return train, hold


def stratified_subsample(df, target_size, seed=0):
    """Stratified subsample of `df` preserving the natural class distribution.
    If target_size >= len(df), returns df unchanged."""
    if target_size >= len(df):
        return df.reset_index(drop=True)
    cls_counts = df["Label"].value_counts()
    total = int(cls_counts.sum())
    target_per_class = {c: int(round(target_size * cls_counts[c] / total))
                        for c in cls_counts.index}
    # Round-off adjustment
    diff = target_size - sum(target_per_class.values())
    if diff != 0:
        target_per_class[cls_counts.idxmax()] += diff
    parts = []
    for c, n in target_per_class.items():
        pool = df[df["Label"] == c]
        n_take = min(n, len(pool))
        parts.append(pool.sample(n=n_take, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


In [8]:
# ============================================================
# 7) ANCHOR WRAPPERS
# ============================================================
def rf_anchor(train_df, test_df, features, seed=0):
    """RandomForest: n_estimators=200, balanced class weights, sqrt feature subsampling."""
    Xtr = train_df[features].values; ytr = train_df["Label"].values
    Xte = test_df[features].values;  yte = test_df["Label"].values
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=None, max_features="sqrt",
        class_weight="balanced", random_state=seed, n_jobs=-1,
    )
    rf.fit(Xtr, ytr)
    yhat = rf.predict(Xte)
    return metrics_from_predictions(yte, yhat,
                                     classes=sorted(set(yte.tolist())),
                                     normal_label=NORMAL_LABEL)


def xgb_anchor(train_df, test_df, features, seed=0):
    """XGBoost: n_estimators=200, max_depth=6, lr=0.1, balanced class handling."""
    Xtr = train_df[features].values; ytr_str = train_df["Label"].values
    Xte = test_df[features].values;  yte_str = test_df["Label"].values
    le = LabelEncoder().fit(np.concatenate([ytr_str, yte_str]))
    ytr = le.transform(ytr_str)
    n_classes = len(le.classes_)
    if n_classes == 2:
        pos_label = 1 if "Attack" in le.classes_ else 1
        n_pos = int((ytr == pos_label).sum()); n_neg = int((ytr == 1-pos_label).sum())
        spw = n_neg / max(1, n_pos)
        clf = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            tree_method="hist", eval_metric="logloss",
            scale_pos_weight=spw, random_state=seed, n_jobs=-1, verbosity=0,
        )
        clf.fit(Xtr, ytr)
    else:
        counts = np.bincount(ytr, minlength=n_classes).astype(float)
        weights_per_class = counts.sum() / (n_classes * np.maximum(counts, 1.0))
        sample_weight = weights_per_class[ytr]
        clf = xgb.XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            tree_method="hist", eval_metric="merror",
            objective="multi:softprob", num_class=n_classes,
            random_state=seed, n_jobs=-1, verbosity=0,
        )
        clf.fit(Xtr, ytr, sample_weight=sample_weight)
    yhat = le.inverse_transform(clf.predict(Xte))
    return metrics_from_predictions(yte_str, yhat,
                                     classes=sorted(le.classes_.tolist()),
                                     normal_label=NORMAL_LABEL)


def tabpfn_anchor(train_df, test_df, features, seed=0):
    """TabPFN v2 via tabpfn-client (cloud API). Matches v8 path-2 (cloud)."""
    if not HAS_TABPFN:
        return None
    Xtr = train_df[features].values; ytr = train_df["Label"].values
    Xte = test_df[features].values;  yte = test_df["Label"].values
    try:
        tabpfn_init()
        clf = TabPFNCloud()
        clf.fit(Xtr, ytr)
        yhat = clf.predict(Xte)
    except Exception as e:
        print(f"  TabPFN cloud failed ({type(e).__name__}: {str(e)[:120]})")
        return None
    return metrics_from_predictions(yte, yhat,
                                     classes=sorted(set(yte.tolist())),
                                     normal_label=NORMAL_LABEL)


def tabicl_anchor(train_df, test_df, features, seed=0):
    """TabICL v2.x (local package) with v8 cell 21 config."""
    if not HAS_TABICL:
        return None
    Xtr = train_df[features].values; ytr = train_df["Label"].values
    Xte = test_df[features].values;  yte = test_df["Label"].values
    device = "cuda" if torch.cuda.is_available() else "cpu"
    try:
        clf = TabICLClassifier(
            n_estimators=16,
            norm_methods=["none", "power"],
            feat_shuffle_method="latin",
            class_shuffle_method="shift",
            outlier_threshold=4.0,
            softmax_temperature=0.9,
            average_logits=True,
            support_many_classes=True,
            batch_size=8,
            use_amp="auto",
            allow_auto_download=True,
            checkpoint_version="tabicl-classifier-v1.1-20250506.ckpt",
            device=device,
            random_state=seed,
            verbose=False,
        )
        clf.fit(Xtr, ytr)
        yhat = clf.predict(Xte)
    except Exception as e:
        print(f"  TabICL v2.x failed ({type(e).__name__}: {str(e)[:120]}); falling back to defaults")
        try:
            clf = TabICLClassifier(device=device, random_state=seed)
            clf.fit(Xtr, ytr)
            yhat = clf.predict(Xte)
        except Exception as e2:
            print(f"  TabICL retry also failed: {e2}")
            return None
    return metrics_from_predictions(yte, yhat,
                                     classes=sorted(set(yte.tolist())),
                                     normal_label=NORMAL_LABEL)


ANCHOR_FNS = {
    "RandomForest": rf_anchor,
    "XGBoost":      xgb_anchor,
    "TabPFN":       tabpfn_anchor,
    "TabICL":       tabicl_anchor,
}
print(f"Anchor wrappers registered: {list(ANCHOR_FNS.keys())}")


Anchor wrappers registered: ['RandomForest', 'XGBoost', 'TabPFN', 'TabICL']


In [9]:
# ============================================================
# 8) LOAD + PREPROCESS + SELECT FEATURES + BUILD SPLITS
# ============================================================
# For each (dataset, task) we build a SINGLE stratified 80/20 split
# (train_pool, holdout) with seed=SEED_DEFAULT. Both sweeps then use this:
#  - Sweep A: K-shot training set = K examples per class drawn from train_pool
#  - Sweep B: max-context training set = appropriate stratified subsample of train_pool
# Holdout is identical for both sweeps so the comparison isolates training budget.
PROCESSED         = {}    # (ds, task) -> processed corpus
SELECTED_FEATURES = {}    # (ds, task) -> [12 feature names]
TRAIN_POOL        = {}    # (ds, task) -> 80% of corpus, balanced strat'd
HOLDOUT           = {}    # (ds, task) -> 20% holdout

print("=" * 60)
print(" LOAD + PREPROCESS + SELECT FEATURES + BUILD SPLITS")
print("=" * 60)
for name in ["swat", "hai", "wustl"]:
    print(f"\n>>> {name.upper()} <<<")
    raw = load_dataset_raw(name)
    proc_b = preprocess(name, raw.copy(), binary=True)
    print(f"  binary preprocess: {proc_b.shape}")
    if len(proc_b) == 0:
        print(f"  WARNING: empty after preprocess, skipping {name}")
        continue
    proc_b = subsample_per_class(proc_b, cap=SUBSAMPLE_PER_CLASS, seed=0)
    PROCESSED[(name, "binary")] = proc_b
    feats = select_top_features(proc_b, k=TOP_K_FEATURES, seed=0)
    SELECTED_FEATURES[(name, "binary")] = feats
    train_pool, holdout = stratified_split(proc_b, train_frac=TRAIN_FRAC, seed=SEED_DEFAULT)
    TRAIN_POOL[(name, "binary")] = train_pool
    HOLDOUT[(name, "binary")] = holdout
    print(f"  selected {len(feats)} features: {feats}")
    print(f"  train_pool: {train_pool.shape}  classes={train_pool['Label'].value_counts().to_dict()}")
    print(f"  holdout:    {holdout.shape}     classes={holdout['Label'].value_counts().to_dict()}")

# WUSTL multi-class
print(f"\n>>> WUSTL (multiclass) <<<")
raw_w = load_dataset_raw("wustl")
proc_mc = preprocess("wustl", raw_w.copy(), binary=False)
print(f"  multiclass preprocess: {proc_mc.shape}")
if len(proc_mc) > 0:
    proc_mc = subsample_per_class(proc_mc, cap=SUBSAMPLE_PER_CLASS, seed=0)
    PROCESSED[("wustl", "multiclass")] = proc_mc
    feats_mc = select_top_features(proc_mc, k=TOP_K_FEATURES, seed=0)
    SELECTED_FEATURES[("wustl", "multiclass")] = feats_mc
    train_pool, holdout = stratified_split(proc_mc, train_frac=TRAIN_FRAC, seed=SEED_DEFAULT)
    TRAIN_POOL[("wustl", "multiclass")] = train_pool
    HOLDOUT[("wustl", "multiclass")] = holdout
    print(f"  selected {len(feats_mc)} features: {feats_mc}")
    print(f"  train_pool: {train_pool.shape}  classes={train_pool['Label'].value_counts().to_dict()}")
    print(f"  holdout:    {holdout.shape}     classes={holdout['Label'].value_counts().to_dict()}")

del raw, raw_w; gc.collect()

print()
print("=" * 60)
print(f" READY: {len(PROCESSED)} (dataset, task) combinations")
print("=" * 60)


 LOAD + PREPROCESS + SELECT FEATURES + BUILD SPLITS

>>> SWAT <<<

[load] swat (vishala28/swat-dataset-secure-water-treatment-system)


100%|██████████| 101M/101M [00:06<00:00, 16.8MB/s]

Extracting files...


  kagglehub OK -> /root/.cache/kagglehub/datasets/vishala28/swat-dataset-secure-water-treatment-system/versions/3
  found 3 CSV file(s): ['attack.csv', 'merged.csv', 'normal.csv']
    attack.csv                                         -> (54621, 54)
    merged.csv                                         -> (1441719, 54)
    normal.csv                                         -> (1387098, 54)
  raw swat: (2883438, 54)
  binary preprocess: (899838, 45)
  selected 12 features: ['AIT201', 'AIT501', 'AIT402', 'LIT301', 'AIT502', 'PIT501', 'PIT503', 'PIT502', 'LIT401', 'FIT503', 'AIT203', 'FIT401']
  train_pool: (80000, 45)  classes={'Normal': 40000, 'Attack': 40000}
  holdout:    (20000, 45)     classes={'Normal': 10000, 'Attack': 10000}

>>> HAI <<<

[load] hai (icsdataset/hai-security-dataset)


100%|██████████| 798M/798M [00:46<00:00, 18.2MB/s]

Extracting files...


  kagglehub OK -> /root/.cache/kagglehub/datasets/icsdataset/hai-security-dataset/versions/10
  HAI version chosen: hai-22.04
  reading 10 HAI CSV file(s)
    test1.csv                      -> (86400, 89)
    test2.csv                      -> (82800, 89)
    test3.csv                      -> (62400, 89)
    test4.csv                      -> (129600, 89)
    train1.csv                     -> (93601, 89)
    train2.csv                     -> (201600, 89)
    train3.csv                     -> (126000, 89)
    train4.csv                     -> (86401, 89)
    train5.csv                     -> (237600, 89)
    train6.csv                     -> (259200, 89)
  raw hai: (1365602, 89)
  binary preprocess: (1365602, 71)
  selected 12 features: ['P1_TIT03', 'P2_AutoSD', 'P1_B2004', 'P1_B3004', 'P1_B3005', 'P4_ST_PS', 'P1_PP04SP', 'P4_HT_PS', 'P2_ManualSD', 'P1_B4002', 'P1_PCV02Z', 'P2_SCO']
  train_pool: (49624, 71)  classes={'Normal': 40000, 'Attack': 9624}
  holdout:    (12406, 71)     classes=

100%|██████████| 105M/105M [00:07<00:00, 15.4MB/s]

Extracting files...


  kagglehub OK -> /root/.cache/kagglehub/datasets/annaamalaiu/wustl-iiot-2021-dataset/versions/1
  found 1 CSV file(s): ['wustl_iiot_2021.csv']
    wustl_iiot_2021.csv                      -> (1194464, 50)
  raw wustl: (1194464, 50)
  binary preprocess: (1194464, 42)
  selected 12 features: ['SrcBytes', 'TotBytes', 'TotAppByte', 'SAppBytes', 'sTtl', 'TotPkts', 'SIntPkt', 'pLoss', 'Load', 'SrcLoad', 'Dur', 'RunTime']
  train_pool: (80000, 42)  classes={'Normal': 40000, 'Attack': 40000}
  holdout:    (20000, 42)     classes={'Normal': 10000, 'Attack': 10000}

>>> WUSTL (multiclass) <<<

[load] wustl (annaamalaiu/wustl-iiot-2021-dataset)
Using Colab cache for faster access to the 'wustl-iiot-2021-dataset' dataset.
  kagglehub OK -> /kaggle/input/wustl-iiot-2021-dataset
  found 1 CSV file(s): ['wustl_iiot_2021.csv']
    wustl_iiot_2021.csv                      -> (1194464, 50)
  raw wustl: (1194464, 50)
  multiclass preprocess: (1194464, 42)
  selected 12 features: ['sTtl', 'pLoss', 'IdleT

In [10]:
# ============================================================
# 9) SWEEP A -- K-SHOT REGIME (matches paper / v1.3)
# ============================================================
# All four anchors trained on 10 examples per class drawn from train_pool.
# Holdout is the (unchanged) 20% partition.
TASKS = [
    ("swat",  "binary"),
    ("hai",   "binary"),
    ("wustl", "binary"),
    ("wustl", "multiclass"),
]

sweep_a_rows = []
sweep_a_perclass = []
print("=" * 60)
print(f" SWEEP A -- K-SHOT REGIME ({K_SHOT_PER_CLASS} per class)")
print("=" * 60)
for ds_name, task in TASKS:
    if (ds_name, task) not in TRAIN_POOL:
        print(f"\n--- {ds_name.upper()} {task} SKIPPED ---")
        continue
    train_pool = TRAIN_POOL[(ds_name, task)]
    holdout    = HOLDOUT[(ds_name, task)]
    features   = SELECTED_FEATURES[(ds_name, task)]
    print(f"\n--- {ds_name.upper()} {task} (features={len(features)}) ---")

    # K-shot ICL set: drawn from train_pool ONCE with seed=SEED_DEFAULT (matches v8)
    icl_df = build_kshot_split(train_pool, features, K_SHOT_PER_CLASS,
                                holdout_df=holdout, seed=SEED_DEFAULT)
    print(f"  K-shot ICL: {icl_df.shape}  classes={icl_df['Label'].value_counts().to_dict()}")
    print(f"  Holdout:    {holdout.shape}")

    for model_name, fn in ANCHOR_FNS.items():
        for seed_i in SEED_LIST:
            t0 = time.time()
            m = fn(icl_df, holdout, features, seed=seed_i)
            wall = time.time() - t0
            if m is None:
                print(f"  [{model_name}] seed {seed_i}: SKIPPED (model unavailable)")
                continue
            row = {
                "regime":    "kshot",
                "dataset":   ds_name, "task": task,
                "model":     model_name, "kind": "Anchor",
                "seed":      seed_i,
                "n_features": len(features),
                "n_train":   len(icl_df),
                "n_test":    len(holdout),
                "accuracy":  m["accuracy"],
                "balanced_accuracy": m["balanced_accuracy"],
                "macro_f1":  m["macro_f1"],
                "MCC":       m["mcc"],
                "FAR":       m["false_alarm_rate"],
                "DR":        m["detection_rate"],
                "wall_s":    wall,
            }
            sweep_a_rows.append(row)
            for cls, pc in m["per_class"].items():
                sweep_a_perclass.append({
                    "regime":   "kshot",
                    "dataset":  ds_name, "task": task,
                    "model":    model_name, "seed": seed_i,
                    "class":    cls,
                    "precision": pc["precision"], "recall": pc["recall"],
                    "f1": pc["f1"], "support": pc["support"],
                })
            print(f"  [{model_name:<13}] seed {seed_i}: "
                  f"acc={m['accuracy']:.4f}  F1={m['macro_f1']:.4f}  "
                  f"MCC={m['mcc']:.4f}  FAR={m['false_alarm_rate']:.4f}  "
                  f"DR={m['detection_rate']:.4f}  ({wall:.1f}s)")

print()
print("=" * 60)
print(f" SWEEP A COMPLETE: {len(sweep_a_rows)} rows")
print("=" * 60)
pd.DataFrame(sweep_a_rows).to_csv(OUT_DIR / "v14_sweep_A_kshot.csv", index=False)
print(f"Wrote: {OUT_DIR / 'v14_sweep_A_kshot.csv'}")


 SWEEP A -- K-SHOT REGIME (10 per class)

--- SWAT binary (features=12) ---
  K-shot ICL: (20, 13)  classes={'Attack': 10, 'Normal': 10}
  Holdout:    (20000, 45)
  [RandomForest ] seed 42: acc=0.8143  F1=0.8118  MCC=0.6457  FAR=0.0710  DR=0.6995  (0.5s)
  [RandomForest ] seed 43: acc=0.8154  F1=0.8126  MCC=0.6501  FAR=0.0635  DR=0.6942  (0.5s)
  [RandomForest ] seed 44: acc=0.8153  F1=0.8125  MCC=0.6498  FAR=0.0637  DR=0.6942  (0.5s)
  [XGBoost      ] seed 42: acc=0.7855  F1=0.7835  MCC=0.5821  FAR=0.1177  DR=0.6888  (0.7s)
  [XGBoost      ] seed 43: acc=0.7855  F1=0.7835  MCC=0.5821  FAR=0.1177  DR=0.6888  (0.2s)
  [XGBoost      ] seed 44: acc=0.7855  F1=0.7835  MCC=0.5821  FAR=0.1177  DR=0.6888  (0.2s)


########  ########   ###  #########  #########       ###         #####     ########  ########
     ###        ##   ###  ###   ###        ###       ###        ###  ###   ##   ###  ###     
########  #######    ###  ###   ###  #######         ###        ########   ######    ########
###       ###   ##   ###  ###   ###  ###   ###       ###        ###  ###   ##   ###       ###
###       ###   ##   ###  #########  ###   ###       ########   ###  ###   ########  ########                      

Thanks for being part of the journey

TabPFN is under active development, please help us improve and report any bugs/ideas you find.

Report issues: https://github.com/priorlabs/tabpfn-client/issues

...

00:01 Predicting... Done!
  [TabPFN       ] seed 44: acc=0.9898  F1=0.9897  MCC=0.9796  FAR=0.0188  DR=0.9983  (2.9s)
  [TabICL       ] seed 42: acc=0.9858  F1=0.9858  MCC=0.9720  FAR=0.0279  DR=0.9995  (2.4s)
  [TabICL       ] seed 43: acc=0.9857  F1=0.9857  MCC=0.9718  FAR=0.0278  DR=0.9992  (2.4s)
  [TabICL       ] seed 44: acc=0.9845  F1=0.9844  MCC=0.9692  FAR=0.0277  DR=0.9966  (2.4s)

--- WUSTL multiclass (features=12) ---
  K-shot ICL: (50, 13)  classes={'Comminj': 10, 'Normal': 10, 'Reconn': 10, 'Dos': 10, 'Backdoor': 10}
  Holdout:    (21742, 42)
  [RandomForest ] seed 42: acc=0.9476  F1=0.7362  MCC=0.9148  FAR=0.0285  DR=0.9991  (0.6s)
  [RandomForest ] seed 43: acc=0.9435  F1=0.7344  MCC=0.9087  FAR=0.0287  DR=0.9991  (0.5s)
  [RandomForest ] seed 44: acc=0.9408  F1=0.7324  MCC=0.9048  FAR=0.0284  DR=0.9991  (0.5s)
  [XGBoost      ] seed 42: acc=0.8816  F1=0.6035  MCC=0.8126  FAR=0.0462  DR=0.9213  (0.3s)
  [XGBoost      ] seed 43: acc=0.8816  F1=0.6035  MCC=0.8126  FAR=0.0

00:00 Fitting... Done!
00:00 Predicting... \

00:02 Predicting... Done!
  [TabPFN       ] seed 43: acc=0.9363  F1=0.6300  MCC=0.8976  FAR=0.0206  DR=0.9997  (3.6s)
00:00 Fitting... \

00:00 Fitting... Done!
00:00 Predicting... \

00:02 Predicting... Done!
  [TabPFN       ] seed 44: acc=0.9363  F1=0.6300  MCC=0.8976  FAR=0.0206  DR=0.9997  (3.0s)
  [TabICL       ] seed 42: acc=0.9701  F1=0.7239  MCC=0.9495  FAR=0.0290  DR=0.9988  (2.6s)
  [TabICL       ] seed 43: acc=0.9742  F1=0.7433  MCC=0.9560  FAR=0.0291  DR=0.9991  (2.6s)
  [TabICL       ] seed 44: acc=0.9757  F1=0.7475  MCC=0.9586  FAR=0.0291  DR=0.9991  (2.6s)

 SWEEP A COMPLETE: 48 rows
Wrote: xgb_outputs_v1_4/v14_sweep_A_kshot.csv


In [11]:
# ============================================================
# 10) SWEEP B -- MAX-CONTEXT REGIME (each method at its native budget)
# ============================================================
sweep_b_rows = []
sweep_b_perclass = []
print("=" * 60)
print(f" SWEEP B -- MAX-CONTEXT REGIME")
print(f"   RF/XGBoost: full train_pool")
print(f"   TabPFN:     {MAX_CTX_TABPFN:,} stratified rows")
print(f"   TabICL:     {MAX_CTX_TABICL:,} stratified rows")
print("=" * 60)

# Per-method max budget. None means "no cap (use full train_pool)".
MAX_BUDGET = {
    "RandomForest": None,
    "XGBoost":      None,
    "TabPFN":       MAX_CTX_TABPFN,
    "TabICL":       MAX_CTX_TABICL,
}

for ds_name, task in TASKS:
    if (ds_name, task) not in TRAIN_POOL:
        print(f"\n--- {ds_name.upper()} {task} SKIPPED ---")
        continue
    train_pool = TRAIN_POOL[(ds_name, task)]
    holdout    = HOLDOUT[(ds_name, task)]
    features   = SELECTED_FEATURES[(ds_name, task)]
    print(f"\n--- {ds_name.upper()} {task} (features={len(features)}, train_pool={len(train_pool):,}, holdout={len(holdout):,}) ---")

    for model_name, fn in ANCHOR_FNS.items():
        budget = MAX_BUDGET[model_name]
        # Subsample train_pool to per-method budget. Fixed seed=SEED_DEFAULT
        # so all seeds for this model see the SAME max-context training set,
        # mirroring the v8 anchor protocol where the ICL split is fixed.
        if budget is None or budget >= len(train_pool):
            train_subset = train_pool
            budget_label = f"full ({len(train_subset):,})"
        else:
            train_subset = stratified_subsample(train_pool, budget, seed=SEED_DEFAULT)
            budget_label = f"{budget:,}"
        print(f"  [{model_name:<13}] training budget = {budget_label} rows; classes={train_subset['Label'].value_counts().to_dict()}")

        for seed_i in SEED_LIST:
            t0 = time.time()
            m = fn(train_subset, holdout, features, seed=seed_i)
            wall = time.time() - t0
            if m is None:
                print(f"  [{model_name:<13}] seed {seed_i}: SKIPPED (model unavailable)")
                continue
            row = {
                "regime":    "maxctx",
                "dataset":   ds_name, "task": task,
                "model":     model_name, "kind": "Anchor",
                "seed":      seed_i,
                "n_features": len(features),
                "n_train":   len(train_subset),
                "n_test":    len(holdout),
                "accuracy":  m["accuracy"],
                "balanced_accuracy": m["balanced_accuracy"],
                "macro_f1":  m["macro_f1"],
                "MCC":       m["mcc"],
                "FAR":       m["false_alarm_rate"],
                "DR":        m["detection_rate"],
                "wall_s":    wall,
            }
            sweep_b_rows.append(row)
            for cls, pc in m["per_class"].items():
                sweep_b_perclass.append({
                    "regime":   "maxctx",
                    "dataset":  ds_name, "task": task,
                    "model":    model_name, "seed": seed_i,
                    "class":    cls,
                    "precision": pc["precision"], "recall": pc["recall"],
                    "f1": pc["f1"], "support": pc["support"],
                })
            print(f"  [{model_name:<13}] seed {seed_i}: "
                  f"acc={m['accuracy']:.4f}  F1={m['macro_f1']:.4f}  "
                  f"MCC={m['mcc']:.4f}  FAR={m['false_alarm_rate']:.4f}  "
                  f"DR={m['detection_rate']:.4f}  ({wall:.1f}s)")

print()
print("=" * 60)
print(f" SWEEP B COMPLETE: {len(sweep_b_rows)} rows")
print("=" * 60)
pd.DataFrame(sweep_b_rows).to_csv(OUT_DIR / "v14_sweep_B_maxctx.csv", index=False)
print(f"Wrote: {OUT_DIR / 'v14_sweep_B_maxctx.csv'}")


 SWEEP B -- MAX-CONTEXT REGIME
   RF/XGBoost: full train_pool
   TabPFN:     10,000 stratified rows
   TabICL:     50,000 stratified rows

--- SWAT binary (features=12, train_pool=80,000, holdout=20,000) ---
  [RandomForest ] training budget = full (80,000) rows; classes={'Normal': 40000, 'Attack': 40000}
  [RandomForest ] seed 42: acc=0.9990  F1=0.9990  MCC=0.9981  FAR=0.0016  DR=0.9997  (3.6s)
  [RandomForest ] seed 43: acc=0.9990  F1=0.9990  MCC=0.9980  FAR=0.0016  DR=0.9996  (3.5s)
  [RandomForest ] seed 44: acc=0.9990  F1=0.9989  MCC=0.9979  FAR=0.0018  DR=0.9997  (3.7s)
  [XGBoost      ] training budget = full (80,000) rows; classes={'Normal': 40000, 'Attack': 40000}
  [XGBoost      ] seed 42: acc=0.9984  F1=0.9984  MCC=0.9968  FAR=0.0026  DR=0.9994  (0.6s)
  [XGBoost      ] seed 43: acc=0.9984  F1=0.9984  MCC=0.9968  FAR=0.0026  DR=0.9994  (0.6s)
  [XGBoost      ] seed 44: acc=0.9984  F1=0.9984  MCC=0.9968  FAR=0.0026  DR=0.9994  (0.6s)
  [TabPFN       ] training budget = 10,000

00:00 Fitting... Done!
00:00 Predicting... \

00:03 Predicting... Done!
  [TabPFN       ] seed 43: acc=0.9969  F1=0.9969  MCC=0.9938  FAR=0.0046  DR=0.9984  (4.6s)
00:00 Fitting... \

00:00 Fitting... Done!
00:00 Predicting... \

00:03 Predicting... Done!
  [TabPFN       ] seed 44: acc=0.9969  F1=0.9969  MCC=0.9938  FAR=0.0046  DR=0.9984  (4.2s)
  [TabICL       ] training budget = 50,000 rows; classes={'Attack': 25000, 'Normal': 25000}
  [TabICL       ] seed 42: acc=0.9985  F1=0.9985  MCC=0.9970  FAR=0.0028  DR=0.9998  (11.5s)
  [TabICL       ] seed 43: acc=0.9986  F1=0.9986  MCC=0.9972  FAR=0.0025  DR=0.9997  (11.3s)
  [TabICL       ] seed 44: acc=0.9984  F1=0.9984  MCC=0.9969  FAR=0.0028  DR=0.9997  (11.3s)

--- HAI binary (features=12, train_pool=49,624, holdout=12,406) ---
  [RandomForest ] training budget = full (49,624) rows; classes={'Normal': 40000, 'Attack': 9624}
  [RandomForest ] seed 42: acc=0.9894  F1=0.9834  MCC=0.9673  FAR=0.0124  DR=0.9971  (1.1s)
  [RandomForest ] seed 43: acc=0.9894  F1=0.9833  MCC=0.9670  FAR=0.0124  DR=0.9967  (1.0s)
  [RandomForest ] seed 44: acc=0.9893  F1=0.9832  MCC=0.9667  FAR=0.0124  DR=0.9963  (1.1s)
  [XGBoost      ] training budget = full (49,624) rows; classes={'No

...

00:03 Predicting... Done!
  [TabPFN       ] seed 44: acc=0.9994  F1=0.9994  MCC=0.9989  FAR=0.0008  DR=0.9997  (4.3s)
  [TabICL       ] training budget = 50,000 rows; classes={'Attack': 25000, 'Normal': 25000}
  [TabICL       ] seed 42: acc=0.9999  F1=0.9999  MCC=0.9998  FAR=0.0000  DR=0.9998  (10.9s)
  [TabICL       ] seed 43: acc=0.9999  F1=0.9998  MCC=0.9997  FAR=0.0001  DR=0.9998  (10.9s)
  [TabICL       ] seed 44: acc=0.9999  F1=0.9999  MCC=0.9998  FAR=0.0000  DR=0.9998  (10.9s)

--- WUSTL multiclass (features=12, train_pool=86,969, holdout=21,742) ---
  [RandomForest ] training budget = full (86,969) rows; classes={'Dos': 40000, 'Normal': 40000, 'Reconn': 6592, 'Comminj': 207, 'Backdoor': 170}
  [RandomForest ] seed 42: acc=0.9998  F1=0.9908  MCC=0.9997  FAR=0.0001  DR=0.9998  (3.7s)
  [RandomForest ] seed 43: acc=0.9999  F1=0.9933  MCC=0.9998  FAR=0.0001  DR=0.9999  (3.6s)
  [RandomForest ] seed 44: acc=0.9998  F1=0.9908  MCC=0.9997  FAR=0.0001  DR=0.9998  (3.7s)
  [XGBoost     

00:00 Fitting... Done!
00:00 Predicting... \

00:03 Predicting... Done!
  [TabPFN       ] seed 43: acc=0.9994  F1=0.9787  MCC=0.9989  FAR=0.0004  DR=0.9992  (4.4s)
00:00 Fitting... \

00:00 Fitting... Done!
00:00 Predicting... \

00:03 Predicting... Done!
  [TabPFN       ] seed 44: acc=0.9994  F1=0.9787  MCC=0.9989  FAR=0.0004  DR=0.9992  (4.4s)
  [TabICL       ] training budget = 50,000 rows; classes={'Normal': 22997, 'Dos': 22996, 'Reconn': 3790, 'Comminj': 119, 'Backdoor': 98}
  [TabICL       ] seed 42: acc=0.9998  F1=0.9906  MCC=0.9996  FAR=0.0004  DR=0.9999  (11.2s)
  [TabICL       ] seed 43: acc=0.9998  F1=0.9931  MCC=0.9997  FAR=0.0004  DR=1.0000  (11.2s)
  [TabICL       ] seed 44: acc=0.9998  F1=0.9931  MCC=0.9997  FAR=0.0004  DR=1.0000  (11.2s)

 SWEEP B COMPLETE: 48 rows
Wrote: xgb_outputs_v1_4/v14_sweep_B_maxctx.csv


In [12]:
# ============================================================
# 11) WRITE COMBINED OUTPUTS + SUMMARY
# ============================================================
df_a = pd.DataFrame(sweep_a_rows)
df_b = pd.DataFrame(sweep_b_rows)
df_combined = pd.concat([df_a, df_b], ignore_index=True)
df_combined.to_csv(OUT_DIR / "v14_combined.csv", index=False)
print(f"Wrote: {OUT_DIR / 'v14_combined.csv'}  ({len(df_combined)} rows)")

# Per-class long format (both sweeps)
pc_combined = pd.concat([pd.DataFrame(sweep_a_perclass),
                          pd.DataFrame(sweep_b_perclass)], ignore_index=True)
pc_wustl = pc_combined[(pc_combined.dataset == "wustl") & (pc_combined.task == "multiclass")]
pc_wustl.to_csv(OUT_DIR / "v14_perclass_wustl.csv", index=False)
print(f"Wrote: {OUT_DIR / 'v14_perclass_wustl.csv'}  ({len(pc_wustl)} rows)")

# Aggregated summary (mean +/- std per regime / model / dataset / task)
agg_cols = ["accuracy", "macro_f1", "MCC", "FAR", "DR"]
agg = (df_combined.groupby(["dataset", "task", "model", "regime"])[agg_cols]
                  .agg(["mean", "std"]).round(4))
agg.to_csv(OUT_DIR / "v14_summary.csv")
print(f"Wrote: {OUT_DIR / 'v14_summary.csv'}")

# Delta table: maxctx - kshot, mean across seeds
mean_kshot = (df_combined[df_combined.regime == "kshot"]
                .groupby(["dataset","task","model"])[agg_cols].mean())
mean_maxctx = (df_combined[df_combined.regime == "maxctx"]
                 .groupby(["dataset","task","model"])[agg_cols].mean())
delta = (mean_maxctx - mean_kshot).round(4)
delta.to_csv(OUT_DIR / "v14_delta.csv")
print(f"Wrote: {OUT_DIR / 'v14_delta.csv'}")
print()

# Side-by-side print
print("=" * 90)
print(" PER-MODEL HEADLINE COMPARISON (mean across 3 seeds)")
print("=" * 90)
pd.set_option("display.width", 200)
print(agg.to_string())
print()
print("=" * 90)
print(" DELTA TABLE (max-context minus K-shot)")
print("=" * 90)
print(delta.to_string())


Wrote: xgb_outputs_v1_4/v14_combined.csv  (96 rows)
Wrote: xgb_outputs_v1_4/v14_perclass_wustl.csv  (120 rows)
Wrote: xgb_outputs_v1_4/v14_summary.csv
Wrote: xgb_outputs_v1_4/v14_delta.csv

 PER-MODEL HEADLINE COMPARISON (mean across 3 seeds)
                                       accuracy         macro_f1             MCC             FAR              DR        
                                           mean     std     mean     std    mean     std    mean     std    mean     std
dataset task       model        regime                                                                                  
hai     binary     RandomForest kshot    0.7801  0.0070   0.7079  0.0107  0.4464  0.0242  0.2076  0.0040  0.7293  0.0296
                                maxctx   0.9894  0.0001   0.9833  0.0001  0.9670  0.0003  0.0124  0.0000  0.9967  0.0004
                   TabICL       kshot    0.8261  0.0012   0.7646  0.0017  0.5562  0.0055  0.1705  0.0040  0.8120  0.0133
                               

In [13]:
# ============================================================
# 12) DOWNLOAD ALL GENERATED FILES
# ============================================================
import os, shutil, glob

BUNDLE = Path("./v14_outputs_bundle.zip")
files = sorted(glob.glob(str(OUT_DIR / "*.csv")))
print(f"Files in {OUT_DIR.resolve()}:")
for f in files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<40s}  {size_kb:>8.1f} KB")
print()

if BUNDLE.exists():
    BUNDLE.unlink()
shutil.make_archive(str(BUNDLE.with_suffix("")), "zip", root_dir=str(OUT_DIR))
print(f"Bundle: {BUNDLE.resolve()}  ({os.path.getsize(BUNDLE)/1024:.1f} KB)")

try:
    from google.colab import files as colab_files
    print("\nTriggering Colab download...")
    colab_files.download(str(BUNDLE))
except ImportError:
    print("\nNot in Colab. Pick up the bundle manually:")
    print(f"  {BUNDLE.resolve()}")


Files in /content/xgb_outputs_v1_4:
  v14_combined.csv                              14.6 KB
  v14_delta.csv                                  0.9 KB
  v14_perclass_wustl.csv                        10.5 KB
  v14_summary.csv                                2.9 KB
  v14_sweep_A_kshot.csv                          7.3 KB
  v14_sweep_B_maxctx.csv                         7.4 KB

Bundle: /content/v14_outputs_bundle.zip  (11.4 KB)

Triggering Colab download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reading the results

The v1.4 sensitivity analysis produces three quantities per (dataset, task, model):

- **K-shot performance** (`regime == "kshot"`): what the method achieves at the paper's E7 budget (10 per class).
- **Max-context performance** (`regime == "maxctx"`): what the method achieves at its native budget.
- **Delta** (`maxctx - kshot`): how much each method gains from its native budget.

Three plausible patterns, with corresponding manuscript interpretations:

| Pattern | What it means | §5.7 framing |
|---|---|---|
| Tabular FMs gain substantially in max-ctx; classical methods saturate; rankings preserved | TabPFN/TabICL benefit from in-context scaling; the K-shot constraint understated their advantage; classical methods reach near-ceiling either way | "Tabular foundation models genuinely benefit from in-context scaling. The K-shot protocol used in the paper headline understates their advantage; with max-context budgets, the gap between tabular FMs and classical anchors widens to N percentage points of MCC." |
| Everything saturates; rankings vanish | The K-shot protocol exposes meaningful method differences that vanish with abundant training data | "With abundant training data every classifier saturates; the foundation-model-versus-classical comparison is operative in the K-shot regime alone." |
| Classical methods saturate but tabular FMs do not improve | Tabular FMs do not benefit from larger context as expected | "TabPFN/TabICL show limited gain from increased in-context budget on this benchmark, suggesting the K-shot protocol is the natural regime for these methods." |

In all three cases, the §5.7 paragraph honestly scopes the paper's central claim. The K-shot headline stands; v1.4 adds the sensitivity context.

## Hooks into the manuscript

- **§5.7 (new) "Max-context sensitivity"**: A short paragraph reporting the v1.4 results, choosing the §5.7 framing from the table above based on the actual numbers.
- **§6.5 Limitations**: Update the existing "K-shot constraint" sentence to point to §5.7 instead of leaving the sensitivity hypothesised.
- **§6.6 Future Work**: Remove the now-fulfilled "Max-context sensitivity" bullet that was added in the earlier revision round.
- **Tables**: No changes to Tables 2 or 5; v1.4 reports a separate sensitivity table (optionally) that lives in §5.7 or in supplementary material.

After the run completes, share the `v14_combined.csv` and `v14_delta.csv` outputs and I will draft the §5.7 paragraph from the actual numbers.
